In [1]:
import requests


# 1. Đưa địa chỉ kho hàng (lấy thông tin 1 sản phẩm)
url = "https://dummyjson.com/products/1"


print(f"Đang cử shipper đến: {url}")


# 2. Dùng Try-Except để bọc lỗi lỡ như đứt cáp quang hoặc sập mạng
try:
    # Lệnh quan trọng nhất: Gõ cửa và chờ tối đa 5 giây
    response = requests.get(url, timeout=5)
   
    # 3. Kiểm tra mã trạng thái (Status Code)
    print(f"Mã trạng thái shipper báo về: {response.status_code}")
   
    if response.status_code == 200:
        print("✅ [THÀNH CÔNG] Kho đang mở cửa, tín hiệu hoàn hảo!")
    elif response.status_code == 404:
        print("❌ [LỖI 404] Sai địa chỉ, không tìm thấy kho hàng.")
    else:
        print(f"⚠️ [CẢNH BÁO] Mã lỗi {response.status_code}: Có vấn đề từ phía kho hàng.")


# Bắt riêng lỗi chờ quá lâu
except requests.exceptions.Timeout:
    print("⏳ [LỖI TIMEOUT] Đợi 5 giây không thấy phản hồi, hủy nhiệm vụ.")
   
# Bắt các lỗi hỏng mạng, sai định dạng link...
except requests.exceptions.RequestException as e:
    print(f"💥 [LỖI MẠNG] Sập kết nối. Chi tiết: {e}")


Đang cử shipper đến: https://dummyjson.com/products/1
Mã trạng thái shipper báo về: 200
✅ [THÀNH CÔNG] Kho đang mở cửa, tín hiệu hoàn hảo!


In [2]:
import requests
# 1. Gõ cửa kho hàng
url = "https://dummyjson.com/products/1"
response = requests.get(url, timeout=5)


if response.status_code == 200:
    # Bước 2: Khui thùng hàng (Biến JSON thành cái tủ "data")
    data = response.json()
   
    # Bước 3: Thử mở ngăn kéo lấy đồ
    ten_san_pham = data["title"]
    gia_tien = data["price"]
   
    print("--- KHUI HÀNG THÀNH CÔNG ---")
    print(f"🎯 Đã nhặt được sản phẩm: {ten_san_pham}")
    print(f"💵 Giá tiền: {gia_tien} USD")
else:
    print("Không lấy được hàng, mã lỗi:", response.status_code)


--- KHUI HÀNG THÀNH CÔNG ---
🎯 Đã nhặt được sản phẩm: Essence Mascara Lash Princess
💵 Giá tiền: 9.99 USD


In [3]:
import requests
url = "https://dummyjson.com/products/1"
response = requests.get(url, timeout=5)


if response.status_code == 200:
    data = response.json()
   
    # --- CÁCH VIẾT CỦA DATA ENGINEER ---
    # Dùng .get() để nhặt đúng 5 trường bạn cần từ con số 0
   
    id_sp = data.get("id")
    ten_sp = data.get("title", "Không có tên") # Nhớ là web này dùng "title" thay vì "name"
    gia_tien = data.get("price", 0.0)          # Nếu thiếu giá, mặc định cho là 0.0
    thuong_hieu = data.get("brand", "Unknown") # Nếu thiếu hãng, ghi là Unknown
    danh_gia = data.get("rating", 0.0)
   
    # Cố tình tìm một trường KHÔNG TỒN TẠI để test sự "chống đạn"
    khuyen_mai = data.get("ma_giam_gia", "Không có mã nào cả")
   
    print("--- BÓC TÁCH THÀNH CÔNG, KHÔNG BỊ SẬP ---")
    print(f"ID: {id_sp}")
    print(f"Tên: {ten_sp}")
    print(f"Giá: {gia_tien}$")
    print(f"Hãng: {thuong_hieu}")
    print(f"Rating: {danh_gia} sao")
    print(f"Mã giảm giá (Test lỗi): {khuyen_mai}")
else:
    print("Không lấy được hàng!")


--- BÓC TÁCH THÀNH CÔNG, KHÔNG BỊ SẬP ---
ID: 1
Tên: Essence Mascara Lash Princess
Giá: 9.99$
Hãng: Essence
Rating: 2.56 sao
Mã giảm giá (Test lỗi): Không có mã nào cả


In [4]:
import requests


# Đây chính là "Tờ hướng dẫn" hoàn hảo để giao cho Spark
def fetch_and_extract(url):
    print(f"Đang cào: {url}") # In ra để lúc test dễ nhìn
   
    try:
        response = requests.get(url, timeout=5)
       
        if response.status_code == 200:
            data = response.json()
            # Trả về từ điển chứa Data sạch
            return {
                "id": data.get("id"),
                "title": data.get("title", "Không có tên"),
                "price": data.get("price", 0.0),
                "brand": data.get("brand", "Unknown"),
                "rating": data.get("rating", 0.0)
            }
        else:
            # Trả về từ điển chứa Lỗi hệ thống
            return {"id": None, "error": f"Lỗi HTTP: {response.status_code}"}
           
    except requests.exceptions.Timeout:
        # Trả về từ điển chứa Lỗi chờ lâu
        return {"id": None, "error": "Lỗi: Quá thời gian chờ (Timeout)"}
       
    except requests.exceptions.RequestException as e:
        # Trả về từ điển chứa Lỗi sập mạng
        return {"id": None, "error": "Lỗi: Sập kết nối mạng"}


# ==========================================
# GÓC NGHIỆM THU (TESTING)
# ==========================================
print("\n--- BẮT ĐẦU NGHIỆM THU ---")


# Test 1: Link ngon (Kỳ vọng: Ra data đẹp)
ket_qua_1 = fetch_and_extract("https://dummyjson.com/products/1")
print("Kết quả 1:", ket_qua_1)


# Test 2: Link tạch (Kỳ vọng: Ra dictionary chứa mã lỗi 404)
ket_qua_2 = fetch_and_extract("https://dummyjson.com/products/99999")
print("Kết quả 2:", ket_qua_2)


# Test 3: Mạng tạch (Kỳ vọng: Ra dictionary báo lỗi sập mạng)
ket_qua_3 = fetch_and_extract("https://trang-web-nay-khong-ton-tai-tren-doi.com")
print("Kết quả 3:", ket_qua_3)



--- BẮT ĐẦU NGHIỆM THU ---
Đang cào: https://dummyjson.com/products/1
Kết quả 1: {'id': 1, 'title': 'Essence Mascara Lash Princess', 'price': 9.99, 'brand': 'Essence', 'rating': 2.56}
Đang cào: https://dummyjson.com/products/99999
Kết quả 2: {'id': None, 'error': 'Lỗi HTTP: 404'}
Đang cào: https://trang-web-nay-khong-ton-tai-tren-doi.com
Kết quả 3: {'id': None, 'error': 'Lỗi: Sập kết nối mạng'}


Đang gọi Quản đốc Spark tới công trường...
✅ Quản đốc Spark đã sẵn sàng làm việc!
🛠️ Phiên bản Spark đang dùng: 3.5.0


📦 Đã chuẩn bị xấp công việc: 10 links.
✂️ Quản đốc Spark đã chia 10 links này thành 4 phần (partitions).
